# HuggingFace Tokenizer 실전

> Phase 1에서 BPE를 직접 구현했다. 이제 실전 토크나이저의 **입출력 형식과 특수 기능**을 익힌다.

In [1]:
from transformers import AutoTokenizer
import torch

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. Phase 1 → Phase 2 연결: 토크나이저 출력 비교

In [ ]:
# === Phase 1의 BasicTokenizer vs HuggingFace Tokenizer ===
# Phase 1: encode("hello") -> [104, 101, 256, 111]  (ID 리스트만 반환)
# Phase 2: tokenizer("hello") -> {input_ids: [...], attention_mask: [...]}

tokenizer = AutoTokenizer.from_pretrained('gpt2')

text = "Hello, how are you?"
output = tokenizer(text)

print(f"입력: '{text}'")
print(f"\n출력 키: {list(output.keys())}")
print(f"  input_ids:      {output['input_ids']}")
print(f"  attention_mask:  {output['attention_mask']}")

print(f"\n비교:")
print(f"  Phase 1 encode(): 토큰 ID 리스트만 반환")
print(f"  Phase 2 tokenizer(): input_ids + attention_mask (모델이 필요한 추가 정보 포함)")

입력: 'Hello, how are you?'

출력 키: ['input_ids', 'attention_mask']
  input_ids:      [15496, 11, 703, 389, 345, 30]
  attention_mask:  [1, 1, 1, 1, 1, 1]

비교:
  Phase 1 encode(): 토큰 ID 리스트만 반환
  Phase 2 tokenizer(): input_ids + attention_mask (모델이 필요한 추가 정보 포함)


In [3]:
# === 각 토큰 ID가 어떤 문자열인지 확인 ===
# Phase 1에서 vocab[idx]로 했던 것과 동일
tokens = tokenizer.tokenize(text)  # 문자열 형태로 토큰화
ids = tokenizer.convert_tokens_to_ids(tokens)  # 문자열 -> ID

print(f"입력: '{text}'")
print(f"토큰 문자열: {tokens}")
print(f"토큰 ID:    {ids}")
print(f"\n각 토큰 확인:")
for tok, tid in zip(tokens, ids):
    print(f"  '{tok}' -> {tid}")

# 'G' 같은 접두사는 BPE에서 단어 중간/끝 토큰을 의미
# GPT-2는 띄어쓰기를 'G'로 표시 (Phase 1에서 바이트로 처리한 것과 동일 원리)

입력: 'Hello, how are you?'
토큰 문자열: ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']
토큰 ID:    [15496, 11, 703, 389, 345, 30]

각 토큰 확인:
  'Hello' -> 15496
  ',' -> 11
  'Ġhow' -> 703
  'Ġare' -> 389
  'Ġyou' -> 345
  '?' -> 30


---
## 2. Special Tokens (특수 토큰)

In [4]:
# === 모델별 특수 토큰 비교 ===
# Phase 1에서 <|endoftext|> 같은 특수 토큰을 개념적으로 배웠다
# 실전에서는 모델마다 다른 특수 토큰을 사용

# GPT-2 특수 토큰
print("GPT-2 특수 토큰:")
print(f"  BOS (\uc2dc\uc791): {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"  EOS (\ub05d):   {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"  PAD (\ud328\ub529): {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

print(f"\n-> GPT-2는 PAD 토큰이 없다! (배치 처리 시 직접 설정 필요)")

GPT-2 특수 토큰:
  BOS (시작): <|endoftext|> (ID: 50256)
  EOS (끝):   <|endoftext|> (ID: 50256)
  PAD (패딩): None (ID: None)

-> GPT-2는 PAD 토큰이 없다! (배치 처리 시 직접 설정 필요)


In [5]:
# === PAD 토큰 설정 ===
# GPT-2처럼 PAD가 없는 모델은 EOS를 PAD로 재사용하는 것이 관례
tokenizer.pad_token = tokenizer.eos_token
print(f"PAD 설정 후: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"-> EOS 토큰을 PAD로 재사용")

PAD 설정 후: <|endoftext|> (ID: 50256)
-> EOS 토큰을 PAD로 재사용


---
## 3. Padding과 Truncation

In [6]:
# === 배치 처리: 길이가 다른 문장들을 한 번에 처리 ===
# 모델은 텐서 연산을 하므로 같은 길이가 필요 -> 패딩으로 맞춤
texts = [
    "Hello world",           # 짧은 문장
    "How are you doing today?",  # 긴 문장
]

# 패딩 없이 토큰화 (길이 다름)
no_pad = tokenizer(texts)
print("패딩 없이:")
for i, text in enumerate(texts):
    print(f"  '{text}' -> {no_pad['input_ids'][i]} (길이: {len(no_pad['input_ids'][i])})")

print()

# 패딩 적용 (같은 길이로 맞춤)
padded = tokenizer(texts, padding=True, return_tensors='pt')
print("패딩 적용 후:")
print(f"  input_ids shape: {padded['input_ids'].shape}")  # [2, 최대길이]
for i, text in enumerate(texts):
    ids = padded['input_ids'][i].tolist()
    mask = padded['attention_mask'][i].tolist()
    print(f"  '{text}'")
    print(f"    input_ids:      {ids}")
    print(f"    attention_mask:  {mask}")
    # attention_mask=0인 위치가 패딩 (PAD)
    print(f"    -> mask=0 위치가 PAD. 모델은 이 위치를 무시함")

패딩 없이:
  'Hello world' -> [15496, 995] (길이: 2)
  'How are you doing today?' -> [2437, 389, 345, 1804, 1909, 30] (길이: 6)

패딩 적용 후:
  input_ids shape: torch.Size([2, 6])
  'Hello world'
    input_ids:      [15496, 995, 50256, 50256, 50256, 50256]
    attention_mask:  [1, 1, 0, 0, 0, 0]
    -> mask=0 위치가 PAD. 모델은 이 위치를 무시함
  'How are you doing today?'
    input_ids:      [2437, 389, 345, 1804, 1909, 30]
    attention_mask:  [1, 1, 1, 1, 1, 1]
    -> mask=0 위치가 PAD. 모델은 이 위치를 무시함


In [7]:
# === Truncation: 너무 긴 문장 자르기 ===
long_text = "This is a very long sentence that we want to truncate " * 10

# max_length=20으로 제한
truncated = tokenizer(
    long_text,
    truncation=True,
    max_length=20,
    return_tensors='pt'
)

print(f"원본 길이: {len(tokenizer.encode(long_text))} 토큰")
print(f"truncation 후: {truncated['input_ids'].shape[1]} 토큰")
print(f"\n-> max_length를 초과하는 토큰은 잘려나감")
print(f"   모델의 최대 시퀀스 길이(GPT-2: 1024)를 초과하면 오류 발생")

원본 길이: 121 토큰
truncation 후: 20 토큰

-> max_length를 초과하는 토큰은 잘려나감
   모델의 최대 시퀀스 길이(GPT-2: 1024)를 초과하면 오류 발생


In [9]:
# === padding_side: 생성 시 left 패딩이 필수인 이유 ===
# Decoder 모델은 오른쪽 끝에서 다음 토큰을 생성
# 오른쪽에 PAD가 있으면 생성이 꼬임

print("Right padding (학습 시):")
print("  [Hello] [world] [PAD] [PAD]  <- PAD가 오른쪽")
print("  Loss 계산 시 PAD 위치는 무시\n")

print("Left padding (생성 시):")
print("  [PAD] [PAD] [Hello] [world]  <- PAD가 왼쪽")
print("  생성: [PAD] [PAD] [Hello] [world] [!]  <- 오른쪽 끝에서 생성\n")

# 설정 방법
tokenizer.padding_side = 'left'  # 생성 시
left_padded = tokenizer(texts, padding=True, return_tensors='pt')
print("Left padding 적용:")
for i, text in enumerate(texts):
    ids = left_padded['input_ids'][i].tolist()
    mask = left_padded['attention_mask'][i].tolist()
    print(f"  '{text}'")
    print(f"    input_ids:      {ids}")
    print(f"    attention_mask:  {mask}")

# 원복
tokenizer.padding_side = 'right'

Right padding (학습 시):
  [Hello] [world] [PAD] [PAD]  <- PAD가 오른쪽
  Loss 계산 시 PAD 위치는 무시

Left padding (생성 시):
  [PAD] [PAD] [Hello] [world]  <- PAD가 왼쪽
  생성: [PAD] [PAD] [Hello] [world] [!]  <- 오른쪽 끝에서 생성

Left padding 적용:
  'Hello world'
    input_ids:      [50256, 50256, 50256, 50256, 15496, 995]
    attention_mask:  [0, 0, 0, 0, 1, 1]
  'How are you doing today?'
    input_ids:      [2437, 389, 345, 1804, 1909, 30]
    attention_mask:  [1, 1, 1, 1, 1, 1]


---
## 4. Chat Template

In [10]:
# === Chat Template: 대화형 모델의 역할 구분 형식 ===
# Phase 1에서 ChatML 형식을 개념적으로 배웠다
# 실전에서는 apply_chat_template()이 자동으로 처리

# 대화 메시지 구성
messages = [
    {"role": "system", "content": "너는 도움이 되는 AI 어시스턴트이다."},
    {"role": "user", "content": "Python으로 Hello World 어떻게 출력해?"},
]

# GPT-2는 chat template이 없으므로, 다른 모델로 시연
# (GPT-2는 대화형이 아닌 순수 텍스트 생성 모델)
print("대화 메시지 구조:")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content']}")

print("\n모델별 변환 예시:")
print("\nChatML (형\uc2dd):")
print("  <|im_start|>system")
print("  \ub108\ub294 \ub3c4\uc6c0\uc774 \ub418\ub294 AI \uc5b4\uc2dc\uc2a4\ud134\ud2b8\uc774\ub2e4.")
print("  <|im_end|>")
print("  <|im_start|>user")
print("  Python\uc73c\ub85c Hello World \uc5b4\ub5bb\uac8c \ucd9c\ub825\ud574?")
print("  <|im_end|>")

print("\nLlama 3 (형\uc2dd):")
print("  <|begin_of_text|><|start_header_id|>system<|end_header_id|>")
print("  \ub108\ub294 \ub3c4\uc6c0\uc774 \ub418\ub294 AI \uc5b4\uc2dc\uc2a4\ud134\ud2b8\uc774\ub2e4.<|eot_id|>")
print("  ...")

print("\n-> apply_chat_template()이 모델에 맞는 형식으로 자동 변환")
print("   직접 형식을 만들 필요 없음!")

대화 메시지 구조:
  [system]: 너는 도움이 되는 AI 어시스턴트이다.
  [user]: Python으로 Hello World 어떻게 출력해?

모델별 변환 예시:

ChatML (형식):
  <|im_start|>system
  너는 도움이 되는 AI 어시스턴트이다.
  <|im_end|>
  <|im_start|>user
  Python으로 Hello World 어떻게 출력해?
  <|im_end|>

Llama 3 (형식):
  <|begin_of_text|><|start_header_id|>system<|end_header_id|>
  너는 도움이 되는 AI 어시스턴트이다.<|eot_id|>
  ...

-> apply_chat_template()이 모델에 맞는 형식으로 자동 변환
   직접 형식을 만들 필요 없음!


---
## 5. 언어별 토큰 효율 비교 (실전)

In [11]:
# === Phase 1에서 tiktoken으로 했던 비교를 HuggingFace로 ===
texts = {
    "영어": "The quick brown fox jumps over the lazy dog.",
    "한국어": "빠른 갈색 여우가 게으른 개를 뚰어넘었다.",
    "Python": "def hello():\n    print('Hello, world!')",
    "숫자": "127 + 399 = 526",
}

print(f"GPT-2 토크나이저 - 언어별 효율:")
print("=" * 60)
for lang, text in texts.items():
    ids = tokenizer.encode(text)
    tokens = tokenizer.tokenize(text)
    n_chars = len(text)
    n_tokens = len(ids)
    print(f"\n{lang}: '{text}'")
    print(f"  토큰: {tokens}")
    print(f"  {n_chars}문자 -> {n_tokens}토큰 (문자당 {n_tokens/n_chars:.2f}토큰)")

print("\n-> 한국어는 영어보다 더 많은 토큰 소비 (Phase 1에서 배운 이유: UTF-8 바이트 수 차이)")

GPT-2 토크나이저 - 언어별 효율:

영어: 'The quick brown fox jumps over the lazy dog.'
  토큰: ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
  44문자 -> 10토큰 (문자당 0.23토큰)

한국어: '빠른 갈색 여우가 게으른 개를 뚰어넘었다.'
  토큰: ['ë', '¹', 'ł', 'ë', '¥', '¸', 'Ġ', 'ê', '°', 'Ī', 'ì', 'ĥ', 'ī', 'Ġì', 'Ĺ', '¬', 'ì', 'ļ', '°', 'ê', '°', 'Ģ', 'Ġ', 'ê', '²', 'Į', 'ì', 'ľ', '¼', 'ë', '¥', '¸', 'Ġ', 'ê', '°', 'ľ', 'ë', '¥', '¼', 'Ġë', 'ļ', '°', 'ì', 'ĸ', '´', 'ë', 'Ħ', 'ĺ', 'ì', 'Ĺ', 'Ī', 'ëĭ', '¤', '.']
  23문자 -> 54토큰 (문자당 2.35토큰)

Python: 'def hello():
    print('Hello, world!')'
  토큰: ['def', 'Ġhello', '():', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġprint', "('", 'Hello', ',', 'Ġworld', '!', "')"]
  39문자 -> 14토큰 (문자당 0.36토큰)

숫자: '127 + 399 = 526'
  토큰: ['127', 'Ġ+', 'Ġ399', 'Ġ=', 'Ġ5', '26']
  15문자 -> 6토큰 (문자당 0.40토큰)

-> 한국어는 영어보다 더 많은 토큰 소비 (Phase 1에서 배운 이유: UTF-8 바이트 수 차이)


In [12]:
# === 디코딩: 토큰 ID -> 텍스트 ===
# Phase 1에서 바이트 이어붙이기 -> UTF-8 디코딩을 했던 것과 동일

text = "Hello, how are you?"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(f"원본:   '{text}'")
print(f"ID:     {ids}")
print(f"복원:   '{decoded}'")
print(f"일치:   {text == decoded}")

# skip_special_tokens: 특수 토큰 제거 후 디코딩
# 생성 결과를 사람에게 보여줄 때 사용
decoded_clean = tokenizer.decode(ids, skip_special_tokens=True)
print(f"\nskip_special_tokens=True: '{decoded_clean}'")

원본:   'Hello, how are you?'
ID:     [15496, 11, 703, 389, 345, 30]
복원:   'Hello, how are you?'
일치:   True

skip_special_tokens=True: 'Hello, how are you?'


---
## 정리

| 개념 | Phase 1 | Phase 2 |
|------|---------|----------|
| **인코딩** | `encode()` → ID 리스트 | `tokenizer()` → input_ids + attention_mask |
| **특수토큰** | 개념만 | BOS/EOS/PAD 실제 설정 |
| **패딩** | 없음 | padding=True, padding_side |
| **대화형식** | ChatML 개념 | apply_chat_template() |